# Pre-processamento das imagens INbreast

## 1. Objetivo do Experimento
**Objetivo:** Gerar regiões candidatas com morfologia compatível com BAC em mamografias INbreast.

**Nomenclatura:** As regiões identificadas serão denominadas **`candidato_BAC`** e nunca `BAC_confirmada`, visto que não possuem confirmação clínica por especialistas.

## 2. Inventário de Arquivos do INbreast
- Foram identificados os arquivos DICOM na pasta `data/external/inbreast/raw/INbreast Release 1.0/AllDICOMs`.
- As anotações originais (XMLs) não foram identificadas na mesma pasta e, portanto, serão extraídas as metainformações dos arquivos DICOM disponíveis (vista, lateralidade, tamanho).

In [1]:
import os
import glob
import pydicom
import pandas as pd
from pathlib import Path

## 3. Exploração dos DICOMs e Manifest
Extraindo informações dos nomes dos arquivos e do Header DICOM para criar um manifesto que contenha:
- `image_id`
- `dcm_path`
- `view`
- `laterality`
- `rows`
- `columns`
- `pixel_spacing_x`
- `pixel_spacing_y`

In [2]:
data_dir = Path('../../data/external/inbreast/raw/INbreast Release 1.0/AllDICOMs')
dcm_files = list(data_dir.glob('*.dcm'))

records = []
for dcm_path in dcm_files:
    try:
        ds = pydicom.dcmread(str(dcm_path), stop_before_pixels=True)
        
        # O identificador no INbreast costuma ser o prefixo do arquivo
        image_id = dcm_path.name.split('_')[0]
        
        # Extraindo vista e lateralidade do nome do arquivo
        parts = dcm_path.stem.split('_')
        if len(parts) >= 5:
            laterality = parts[3]
            view = parts[4]
        else:
            laterality = 'UNKNOWN'
            view = 'UNKNOWN'
        
        rows = getattr(ds, 'Rows', None)
        columns = getattr(ds, 'Columns', None)
        
        pixel_spacing = getattr(ds, 'PixelSpacing', [None, None])
        ps_x = pixel_spacing[0] if pixel_spacing and len(pixel_spacing) > 0 else None
        ps_y = pixel_spacing[1] if pixel_spacing and len(pixel_spacing) > 1 else None
        
        records.append({
            'image_id': image_id,
            'dcm_path': str(dcm_path),
            'view': view,
            'laterality': laterality,
            'rows': rows,
            'columns': columns,
            'pixel_spacing_x': ps_x,
            'pixel_spacing_y': ps_y
        })
    except Exception as e:
        print(f"Erro ao ler {dcm_path.name}: {e}")

df_manifest = pd.DataFrame(records)
print(f"Total de imagens processadas: {len(df_manifest)}")
df_manifest.head()

Total de imagens processadas: 410


,image_id,dcm_path,view,laterality,rows,columns,pixel_spacing_x,pixel_spacing_y
0,22670465,../../data/external/inbreast/raw/INbreast Rele...,CC,L,3328,2560,None,None
1,22678646,../../data/external/inbreast/raw/INbreast Rele...,CC,L,3328,2560,None,None
2,53580885,../../data/external/inbreast/raw/INbreast Rele...,CC,R,4084,3328,None,None
3,50997515,../../data/external/inbreast/raw/INbreast Rele...,CC,R,4084,3328,None,None
4,20586934,../../data/external/inbreast/raw/INbreast Rele...,CC,L,4084,3328,None,None


## 4. Seleção do Lote Inicial para Desenvolvimento
Separar um lote pequeno de imagens da vista ML (Mediolateral) para o desenvolvimento e depuração do fluxo de geração de candidatos BAC. O restante será reservado para posterior processamento.

In [3]:
# Filtrando apenas ML
df_ml = df_manifest[df_manifest['view'] == 'ML'].copy()

if df_ml.empty:
    print("Nenhuma imagem com a vista 'ML' encontrada!")
else:
    # Selecionar lote inicial de 5 imagens
    df_batch1 = df_ml.sample(n=min(5, len(df_ml)), random_state=42).copy()

    # Diretório de saída do manifesto
    output_dir = Path('../../data/interim')
    output_dir.mkdir(parents=True, exist_ok=True)

    manifest_path = output_dir / 'inbreast_batch1_ml_manifest.csv'
    df_batch1.to_csv(manifest_path, index=False)

    print(f"Manifesto do lote 1 salvo em: {manifest_path}")
    print("Imagens selecionadas para o lote inicial:")
    display(df_batch1[['image_id', 'view', 'laterality']])

Manifesto do lote 1 salvo em: ../../data/interim/inbreast_batch1_ml_manifest.csv
Imagens selecionadas para o lote inicial:


,image_id,view,laterality
31,51049276,ML,L
20,50996709,ML,R
402,20587928,ML,R
165,53587104,ML,L
137,20588216,ML,L


## 5. Registro de Execução
**Versão do código:** 1.0 (Inicial)
**Parâmetros:** `random_state=42`, `n=5`, vista `ML`.
**Imagens usadas:** (Listadas acima).